# Pandas II: Combining and Summarising Trade Tables

In the previous notebook, we worked with one annual trade table: we added columns, classified exports, calculated country means, and saved the result.

Now we will bring information from several tables together. Our question is: **how can we combine annual trade data, attach country information, and summarise the result?**

We will:

1. Concatenate two annual tables.
2. Merge country information into the combined table.
3. Explore export values with descriptive statistics.
4. Calculate summaries by country and year.
5. Save our results to Excel.

> **Note**
>
> All export figures in these practice files are **invented for teaching**. Each row represents exports from one country to one partner in one year. Values are in **millions of US dollars**.

## Set up this notebook

Run the imports below. The worked examples use the 2022 and 2023 practice files in `Lecture_5/tables`. The final exercise also uses the 2024 and 2025 files in that folder. Run it with **`Lecture_5`** as the working directory.

In [ ]:
from pathlib import Path
import pandas as pd

## 1. Concatenate two annual tables

We now want observations from **both 2022 and 2023** in one table.

**Concatenation** puts the rows of one table underneath the rows of another. We use it here because both files record the same kinds of information for different years.

Read the two source files into separate variables. Both files have the same three columns: `country`, `partner`, and `export_millions_usd`. Their export values differ, but neither file contains a `year` column. The year appears only in the filename.

In [ ]:
trade_2022 = pd.read_excel(Path("tables") / "trade_2022.xlsx")
trade_2023 = pd.read_excel(Path("tables") / "trade_2023.xlsx")

In [ ]:
trade_2022

In [ ]:
trade_2023

**Predict:** if we concatenate these tables now, how will we know which observations belong to 2022 and which belong to 2023?

### Add the year before combining

Reuse the fixed-value assignment from notebook 4. Add a `year` column to each table: fill it with `2022` in `trade_2022` and `2023` in `trade_2023`.

Try writing the two instructions before opening the solution.

In [ ]:
# Add the year column to trade_2022 and trade_2023.
# Display each table to check your result.

<details>
<summary>Show solution: add the year</summary>

```python
trade_2022["year"] = 2022
trade_2023["year"] = 2023
```

Each instruction fills the new column with the same year in every row. These assignments change the tables in Python's memory; the Excel files remain unchanged.

</details>

Run these preparation instructions before continuing:

**Checkpoint:** each table now has four columns, including `year`. All rows in `trade_2022` should show 2022; all rows in `trade_2023` should show 2023.

### Concatenate the prepared tables
Pass a **list of tables** to `pd.concat()`. Here the square brackets construct a list containing our two DataFrames.

`ignore_index=True` gives the result a fresh numbered index.

In [ ]:
combined_trade = pd.concat([trade_2022, trade_2023], ignore_index=True)
combined_trade

**Checkpoint:** the result has 12 rows, with index labels from 0 to 11. The `year` column identifies the annual source of each observation.

> **Note**
>
>For this operation, pandas aligns columns **by their names**. Check spelling and units before combining tables. Different column names would create separate columns, with missing values where information is unavailable.<br>
<br>
>Concatenation does not match countries or remove duplicate observations. If we accidentally supplied the same table twice, its observations would appear twice.

### Apply our preparation steps to the combined table

Reuse the preparation rules from the previous notebook: add `trade_flow` and classify exports above 500 as `high`, and exports of 500 or below as `low`. Apply the rules to the combined table so both years receive the same treatment.

In [ ]:
combined_trade["trade_flow"] = "exports"
combined_trade.loc[combined_trade["export_millions_usd"] <= 500, "export_category"] = "low"
combined_trade.loc[combined_trade["export_millions_usd"] > 500, "export_category"] = "high"
combined_trade

**Check together:** Germany's export value of 500 in 2022 is `low`, while its value of 550 in 2023 is `high`.

## 2. Merge country information into our table

Now we want to attach information about each exporting country.

This is a **join**: match records using a shared identifier, called a **key**. We will perform it using `pd.merge()`.

| Operation | What we want to do | Our example |
|---|---|---|
| Concatenate | Add observations from another table | Stack the 2022 and 2023 rows |
| Merge | Match related information using a key | Attach the exporting country's region |

Create a small lookup table with **one row per listed country**. Deliberately leave Chile out so we can see what happens when a trade country has no lookup match. `pd.DataFrame()` builds a table from a dictionary: the dictionary keys become column names and each list supplies that column's values.

The listed countries appear in a different order from the trade table.

In [ ]:
country_info = pd.DataFrame({
    "country": ["Germany", "Switzerland"],
    "region": ["Europe", "Europe"]
})
country_info

**Predict:** should the region depend on a country's row position or on its name?

Merge the tables using `country` as the matching key:

- The first table, `combined_trade`, is the **left** table.
- The second table, `country_info`, is the **right** table.
- `on="country"` tells pandas which column to match in both tables.
- `how="left"` keeps every observation from the left table and attaches matching information from the right table.
- `validate="many_to_one"` checks that the right table has at most one row for each country. Many trade observations may match the same lookup row.

In [ ]:
trade_with_regions = pd.merge(
    combined_trade,
    country_info,
    on="country",
    how="left",
    validate="many_to_one"
)
trade_with_regions

**Checkpoint:** we still have 12 trade observations, now with an extra `region` column. Switzerland and Germany receive `Europe`. Chile's four trade observations remain, but their region is missing because Chile is absent from the lookup.

The region describes the **exporting country**, not its partner. Switzerland's lookup row supplies the region for all four of its trade observations.

For this example, country names are our key. In larger datasets, consistent country codes are often a better choice. If the information varied by year, we would need to match both country and year.

The validation checks lookup-key uniqueness. It does not check whether every trade country found a match.

### Check for unmatched countries

In a left join, a country missing from the lookup remains in the result, but its region is missing.

Select the `region` column, use `.isna()` to identify missing values, and use `.sum()` to count them. For this count, `True` contributes 1 and `False` contributes 0.

In [ ]:
trade_with_regions["region"].isna()

In [ ]:
trade_with_regions["region"].isna().sum()

**Checkpoint:** the result is **4**. These are Chile's two partner observations in each of the two years. We are counting missing region cells, not distinct countries.

The left join preserves every trade observation, including those without a lookup match. A missing region does not mean that the export value is missing.

### Compare with an inner join

**Predict:** if we change `how="left"` to `how="inner"`, what will happen to Chile's observations?

An **inner join** keeps only keys found in both tables.

**Try it:** adapt the previous merge to perform an inner join between `combined_trade` and `country_info`. Match on `country` and keep the same validation. Store the result in **`trade_inner`** and display it.

Write and run your code before opening the solution. Use this variable name so the comparison cell below can use your result.

In [ ]:
# Create trade_inner using an inner join, then display it.

<details>
<summary>Show solution: inner join</summary>

```python
trade_inner = pd.merge(
    combined_trade,
    country_info,
    on="country",
    how="inner",
    validate="many_to_one"
)
trade_inner
```

Only `how` changes from the left-join example. The result is saved separately to preserve `trade_with_regions`.

</details>

**Checkpoint:** the inner join contains **8 rows**: four for Switzerland and four for Germany. Chile's four observations are excluded because its country key is absent from the lookup.

Compare the observation counts:

In [ ]:
print("Left join observations:", len(trade_with_regions))
print("Inner join observations:", len(trade_inner))

The inner join has no missing regions **in this example**. It keeps matching keys, not necessarily complete rows. If Chile appeared in the lookup but its region cell were empty, Chile would still match and remain in the inner join with a missing region.

Use a left join when you want to preserve all trade observations and inspect missing lookup information. Use an inner join when your question requires observations with matching keys in both tables.

### See the different join types

The shaded areas show which **country keys** each join keeps. All four options use `pd.merge()`; the `how` argument chooses the join type.

![Left, inner, right, and outer joins illustrated with country keys](assets/merge_types.png)

For this illustration, imagine that the left table contains Chile, Switzerland, and Germany, while the right table contains Switzerland, Germany, and France. These deliberately different country lists let us see what happens to unmatched keys. Our actual `country_info` table contains only Switzerland and Germany; France is included in the diagram to illustrate keys found only on the right.

A retained key without a match receives missing values in the columns coming from the other table. The circles show distinct keys, not the number of result rows: one country can have several trade observations.

**Predict:** which joins retain Chile? Which retain France? Which retain only Switzerland and Germany?

Diagram inspired by the [DAVPy join illustration](https://raw.githubusercontent.com/guiwitz/DAVPy/master/images/left_right.jpeg), redrawn for our trade example.

## 3. Get an overview with descriptive statistics

The new method **`.describe()`** produces several summary statistics at once.

First select the export column so the output focuses on export values. Although `year` is numeric too, an average year is not the statistic we are looking for.

Continue with **`trade_with_regions`**, the left-join result containing all 12 observations. The separate inner-join result is for comparison. Missing regions do not remove the export values from this summary.

In [ ]:
trade_with_regions["export_millions_usd"].describe()

Read the output as follows:

| Label | Meaning |
|---|---|
| `count` | Number of non-missing observations |
| `mean` | Arithmetic average |
| `std` | Sample standard deviation, a measure of spread |
| `min` | Smallest value |
| `25%` | First quartile, or 25th percentile |
| `50%` | Median, or 50th percentile |
| `75%` | Third quartile, or 75th percentile |
| `max` | Largest value |

The percentiles describe positions in the ordered values. They are not percentages of total exports. Pandas may interpolate between observations to calculate them.

**Checkpoint:** there are 12 non-missing observations. The minimum is 200, the maximum is 950, and the mean is approximately 591.67 million US dollars.

These statistics pool both years, all three exporters, and their selected partners. The mean is an average **per recorded observation**. It is not the average of countries' worldwide exports.

**Discuss:** what does the median tell us? What information about countries and years is hidden by this single overall summary?

## 4. Summarise exports by country

Instead of pooling everything, we can calculate a separate result for each country.

`groupby("country")` groups observations with the same country name. We then select the export column and calculate its mean.

In [ ]:
country_mean = trade_with_regions.groupby("country")["export_millions_usd"].mean()
country_mean

Read this from left to right: **group by country**, select **export values**, calculate the **mean**.

**Checkpoint:** Chile's mean is 425, Germany's is 725, and Switzerland's is 625 million US dollars. Each country's result averages four partner-year observations across 2022 and 2023.

`country_mean` is a **Series**, a one-dimensional labelled result. Our original DataFrame has not been replaced.

In [ ]:
type(country_mean)

### Keep the years separate

To answer a more specific question, group by **both country and year**. Pass their column names as a list.

Use `as_index=False` to keep those grouping columns as ordinary columns in the resulting DataFrame.

In [ ]:
country_year_mean = trade_with_regions.groupby(
    ["country", "year"], as_index=False
)["export_millions_usd"].mean()
country_year_mean

**Checkpoint:** Switzerland's mean is 600 in 2022 and 650 in 2023. Each mean averages its two recorded partner observations for that year.

**Predict:** how would the meaning change if we replaced `.mean()` with `.sum()`?

In [ ]:
country_year_sum = trade_with_regions.groupby(
    ["country", "year"], as_index=False
)["export_millions_usd"].sum()
country_year_sum

**Checkpoint:** Switzerland's sum is 1,200 in 2022 and 1,300 in 2023, in millions of US dollars.

A **mean** answers “what is the average value per recorded partner observation?” A **sum** answers “what is the combined value of the recorded observations?”

Our files contain selected partners only. These sums describe the practice data, not total exports to the world.

**Try it:** calculate mean exports for each year across the countries and partners in the dataset.

In [ ]:
# Group by year, select export values, and calculate the mean.

## 5. Save our results

Save the combined, prepared table and the country-year means as separate Excel files inside an `outputs` folder.

First create the folder if it does not already exist.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

Use `.to_excel()` to write each DataFrame. `index=False` prevents pandas from adding its numbered row labels as an extra Excel column.

In [ ]:
trade_with_regions.to_excel(output_dir / "trade_2022_2023_prepared.xlsx", index=False)
country_year_mean.to_excel(output_dir / "country_year_mean.xlsx", index=False)

Open the files in `Lecture_5/outputs` and inspect the results:

- `trade_2022_2023_prepared.xlsx` contains 12 observations, both years, the export categories, and the available exporting-country regions. Chile's four region cells are blank because the lookup has no Chile row.
- `country_year_mean.xlsx` contains six country-year means.

The source files in `tables` remain unchanged. Running the saving cell again replaces these output files. Close them in Excel before saving again, particularly on Windows.

## Review together

1. Why did we concatenate the annual tables but merge the region table?
2. Which column told pandas how to match the country information?
3. Why does the left join retain Chile while the inner join excludes it? Does an inner join guarantee that every cell is filled?
4. What is the difference between the overall mean, a country mean, and a country-year mean?
5. Why are the sums in this notebook not worldwide export totals?

In the next lecture, we can use visualisations to explore differences across countries and changes between years.

## A small reference card

| What do I want to do? | Command |
|---|---|
| Read an Excel table | `pd.read_excel(input_path)` |
| Stack tables vertically | `pd.concat([trade_2022, trade_2023], ignore_index=True)` |
| Attach country information | `pd.merge(combined_trade, country_info, on="country", how="left", validate="many_to_one")` |
| Count missing regions | `trade_with_regions["region"].isna().sum()` |
| Describe export values | `trade_with_regions["export_millions_usd"].describe()` |
| Calculate country means | `trade_with_regions.groupby("country")["export_millions_usd"].mean()` |
| Calculate country-year sums | `trade_with_regions.groupby(["country", "year"], as_index=False)["export_millions_usd"].sum()` |
| Save without the row index | `trade_with_regions.to_excel(output_path, index=False)` |

The names before the dots refer to our tables or results. You can choose other variable names, but you must use them consistently.

## 6. Exercise: Extend the analysis to four years

Combine the trade data for **2022, 2023, 2024, and 2025**, attach each exporting country's region, and calculate mean exports by country and year.

Use the four Excel files in `tables` and the `country_info` lookup created earlier in this notebook. Use new variable names so your exercise does not replace the worked examples.

The loop connects three ideas we have already used: reading a file, adding a fixed-value column, and appending an item to a list. Here, the items we collect are **DataFrames**, rather than filenames.

Try each part before opening its solution. The solutions are collapsed below each task; expand them when you want to check your work.

### A. Collect the tables and concatenate them

The list below pairs each filename with its year. In `for filename, year in annual_files`, Python takes one pair at a time and assigns its two values to `filename` and `year`.

Complete the loop to:

1. Read the file identified by `filename` into `annual_trade`.
2. Add a `year` column filled with the current `year`.
3. Append that DataFrame to `trade_tables`.
4. **After the loop**, concatenate the tables into `all_trade`, with a fresh numbered index.

Add the year **inside the loop, before appending**. Keep concatenation outside the loop: collect the four tables, then combine them once.

**Hint:** use `Path("tables") / filename`, `pd.read_excel()`, fixed-value column assignment, and `.append()`. Indentation determines which instructions belong to the loop.

In [ ]:
annual_files = [
    ("trade_2022.xlsx", 2022),
    ("trade_2023.xlsx", 2023),
    ("trade_2024.xlsx", 2024),
    ("trade_2025.xlsx", 2025)
]

trade_tables = []

for filename, year in annual_files:
    # Read this file into annual_trade.
    # Add the year column using the current year.
    # Append annual_trade to trade_tables.
    pass  # Replace this placeholder with your three instructions.

# Concatenate trade_tables into all_trade after the loop.
# Display all_trade.

**Checkpoint:** `trade_tables` should contain four DataFrames. The combined table should contain **24 rows**, with the three source columns plus the new `year` column, covering 2022 to 2025.

If you run the loop again, also run `trade_tables = []` so you start with an empty list.

<details>
<summary>Show solution A: add years, collect, and concatenate</summary>

```python
annual_files = [
    ("trade_2022.xlsx", 2022),
    ("trade_2023.xlsx", 2023),
    ("trade_2024.xlsx", 2024),
    ("trade_2025.xlsx", 2025)
]

trade_tables = []

for filename, year in annual_files:
    annual_trade = pd.read_excel(Path("tables") / filename)
    annual_trade["year"] = year
    trade_tables.append(annual_trade)

all_trade = pd.concat(trade_tables, ignore_index=True)
all_trade
```

Each iteration reads one file, records its year in every row, and adds its DataFrame to the list. `pd.concat()` receives the completed list after the loop. `trade_tables` is already a list, so it does not need another pair of square brackets.

```python
len(trade_tables)  # 4 tables
len(all_trade)     # 24 observations
```

</details>

### B. Attach the country regions

Use `pd.merge()` to attach `country_info` to `all_trade`. Store the result in `all_trade_with_regions`.

Use a **left join**, with the same matching key and validation as in the worked example. The lookup still omits Chile. Perform the merge **once, after concatenation**, outside the loop.

Check whether any observations have a missing region.

In [ ]:
# Merge all_trade with country_info into all_trade_with_regions.
# Display the result.
# Count missing values in the region column.

**Checkpoint:** the result should still contain **24 rows**, now with five columns. There should be **8 missing regions**: Chile has two partner observations in each of four years. You do not need to add `trade_flow` or `export_category` for this exercise.

<details>
<summary>Show solution B: attach regions</summary>

```python
all_trade_with_regions = pd.merge(
    all_trade,
    country_info,
    on="country",
    how="left",
    validate="many_to_one"
)
all_trade_with_regions
```

The country lookup is the same for all four years, so we can attach it to the combined data in one operation. The left join retains Chile's eight observations with missing regions.

```python
len(all_trade_with_regions)  # 24 observations
all_trade_with_regions["region"].isna().sum()  # 8 missing regions (Chile)
```

</details>

### C. Summarise exports by country and year

Calculate mean exports for each country in each year. Store the result in `all_country_year_mean`, keeping `country` and `year` as ordinary columns.

**Hint:** adapt the earlier country-year summary to use your exercise DataFrame.

**Checkpoint:** there should be **12 rows**: three countries across four years. Switzerland's mean should be **600** in 2022 and **750** in 2025, in millions of US dollars.

Use the left-join result so Chile remains in the summary even though its region is missing.

In [ ]:
# Calculate all_country_year_mean and display it.

<details>
<summary>Show solution C: country-year means</summary>

```python
all_country_year_mean = all_trade_with_regions.groupby(
    ["country", "year"], as_index=False
)["export_millions_usd"].mean()
all_country_year_mean
```

| Country | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|
| Chile | 400 | 450 | 500 | 550 |
| Germany | 700 | 750 | 800 | 860 |
| Switzerland | 600 | 650 | 700 | 750 |

This reference table presents years side by side to help you check your answers. Your DataFrame has one row for each country-year pair. Each mean averages two recorded partner observations; it does not measure total exports to the world.

</details>